# Building the DRS × plastic-recycling panel dataset

**Research question:** Did introducing a deposit return system (DRS) for single-use drink containers increase plastic packaging recycling rates in European countries?

**Output format: a tidy country–year panel (long format), one row per country per year, saved as CSV.**
This is the layout that works for everything you plan to do: univariate summaries (one column at a time), bivariate comparisons (one column against another, or DRS vs non-DRS), and later pre/post or difference-in-differences models, which all expect one row per unit and time period.

**Inputs** (put them in the folder set by `DATA_DIR`):
- `env_waspacr_defaultview_jsonstat.json`: Eurostat plastic packaging recycling rates
- `env_waspac__custom_22962277_jsonstat.json`: Eurostat plastic packaging generated / recycled (kg per capita)
- DRS introduction years: typed in by hand below, from the Reloop deck and our earlier searches. **These must be verified against primary sources** (see the DRS table cell).

In [8]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")    # folder containing the two JSON files
OUT_DIR = Path("data")     # where the CSVs are written

RATES_FILE = DATA_DIR / "env_waspacr_defaultview_jsonstat.json"
WASTE_FILE = DATA_DIR / "env_waspac__custom_22962277_jsonstat.json"

# Aggregates / near-empty geographies left out of the country panel
DROP_GEOS = ["EU27_2020", "UKN"]   # EU27 total; Northern Ireland (rate only for 2022-23)

PLASTIC = "W150102"                # Eurostat code: plastic packaging
RATE_UNIT = "RT_TGT2025"           # rate on the 2025+ target basis (the 2008 one is nearly empty)

## 1. Read the Eurostat JSON-stat files

In [9]:
def parse_jsonstat(path):
    """Turn a Eurostat JSON-stat 2.0 file into a long DataFrame:
    one column per dimension + 'value' + 'flag' (Eurostat status code)."""
    with open(path, encoding="utf-8") as f:
        d = json.load(f)

    ids, size = d["id"], d["size"]
    codes, labels = [], {}
    for dim in ids:
        cat = d["dimension"][dim]["category"]
        idx = cat["index"]
        order = sorted(idx, key=idx.get) if isinstance(idx, dict) else list(idx)
        codes.append(order)
        labels[dim] = cat.get("label", {})

    n = int(np.prod(size))
    values = np.full(n, np.nan)
    v = d["value"]
    if isinstance(v, dict):                      # sparse form: {flat_index: value}
        for k, x in v.items():
            values[int(k)] = x
    else:                                        # dense form: list with nulls
        values[:] = [np.nan if x is None else x for x in v]

    flags = np.full(n, "", dtype=object)
    s = d.get("status", {})
    if isinstance(s, dict):
        for k, x in s.items():
            flags[int(k)] = x
    elif isinstance(s, list):
        for k, x in enumerate(s):
            flags[k] = x or ""

    index = pd.MultiIndex.from_product(codes, names=ids)
    df = pd.DataFrame({"value": values, "flag": flags}, index=index).reset_index()
    return df, labels


rates_raw, rates_labels = parse_jsonstat(RATES_FILE)
waste_raw, waste_labels = parse_jsonstat(WASTE_FILE)
print("rates:", rates_raw.shape, "| waste:", waste_raw.shape)

rates: (3200, 7) | waste: (3200, 8)


### 1a. Outcome variable: plastic packaging recycling rate (%)

In [10]:
rates = (
    rates_raw[(rates_raw["waste"] == PLASTIC) & (rates_raw["unit"] == RATE_UNIT)]
    .rename(columns={"value": "plastic_recycling_rate", "flag": "rate_flag"})
    [["geo", "time", "plastic_recycling_rate", "rate_flag"]]
    .assign(year=lambda x: x["time"].astype(int))
    .drop(columns="time")
)
rates.head()

,geo,plastic_recycling_rate,rate_flag,year
1600,EU27_2020,39.8,,2014
1601,EU27_2020,40.5,,2015
1602,EU27_2020,42.4,,2016
1603,EU27_2020,41.7,,2017
1604,EU27_2020,41.4,,2018


### 1b. Plastic packaging generated and recycled (kg per capita)
Includes where the recycling happened: in the country itself, elsewhere in the EU, or outside the EU. Those columns only exist for some country-years.

In [11]:
op_names = {
    "GEN": "plastic_generated_kg_hab",
    "RCY": "plastic_recycled_kg_hab",
    "RCY_NAT": "recycled_in_country_kg_hab",
    "RCY_EU_FOR": "recycled_other_eu_kg_hab",
    "RCY_NEU": "recycled_outside_eu_kg_hab",
}
waste = (
    waste_raw[(waste_raw["waste"] == PLASTIC) & (waste_raw["unit"] == "KG_HAB")]
    .assign(year=lambda x: x["time"].astype(int), col=lambda x: x["wst_oper"].map(op_names))
    .pivot_table(index=["geo", "year"], columns="col", values="value", aggfunc="first")
    .reset_index()
)
waste.columns.name = None
waste.head()

,geo,year,plastic_generated_kg_hab,plastic_recycled_kg_hab,recycled_in_country_kg_hab,recycled_other_eu_kg_hab,recycled_outside_eu_kg_hab
0,AT,2014,34.16,11.47,NaN,NaN,NaN
1,AT,2015,34.12,11.46,NaN,NaN,NaN
2,AT,2016,34.09,11.45,NaN,NaN,NaN
3,AT,2017,34.36,11.48,NaN,NaN,NaN
4,AT,2018,34.16,10.89,10.35,0.54,0.0


## 2. Treatment variable: when did each country introduce a DRS?

⚠️ **Verify before you rely on this table.** It is typed in by hand:
- `drs_start_deck` is the year in the Reloop slide deck (an advocacy source; its own map marks several countries as "planned").
- `drs_start_used` is the year this notebook uses. It differs from the deck where our earlier searches or my own knowledge disagree (Malta, Portugal, Romania). **None of them is verified yet**: check each against the national scheme operator or a primary source, then edit `drs_start_used` and set `verified = True`.
- Countries whose DRS starts after 2023 (Ireland, Hungary, Poland, Austria, Portugal) are in the table so they are correctly treated as *not yet having a DRS* within your 2014–2023 data.
- A scheme launched mid-year only partly affects that year, so the panel also has a `drs_full_year` flag (first full year = start year + 1).

In [12]:
drs = pd.DataFrame(
    [
        # geo, deck year, year used, note
        ("SE", 1984, 1984, "Deck: 1984 (cans first; check when PET joined)"),
        ("IS", 1989, 1989, ""),
        ("FI", 1996, 1996, ""),
        ("NO", 1999, 1999, ""),
        ("DK", 2002, 2002, ""),
        ("DE", 2003, 2003, ""),
        ("EE", 2005, 2005, ""),
        ("NL", 2005, 2005, "Initially large PET bottles only; check when small bottles and cans were added"),
        ("HR", 2006, 2006, ""),
        ("LT", 2016, 2016, "In-window switcher"),
        ("MT", 2020, 2022, "Deck says 2020; I believe go-live was late 2022. VERIFY"),
        ("SK", 2022, 2022, "In-window switcher (launched early 2022)"),
        ("LV", 2022, 2022, "In-window switcher (launched early 2022)"),
        ("RO", 2022, 2023, "Deck says 2022; I believe go-live was end of 2023. VERIFY"),
        ("PT", 2022, 2026, "Deck says 2022; an earlier source we found says 2026. VERIFY"),
        ("IE", None, 2024, "Not in deck; believed to start in 2024. VERIFY"),
        ("HU", None, 2024, "Not in deck; believed to start in 2024. VERIFY"),
        ("PL", None, 2025, "Not in deck; believed to start in 2025. VERIFY"),
        ("AT", None, 2025, "Not in deck; believed to start in 2025. VERIFY"),
    ],
    columns=["geo", "drs_start_deck", "drs_start_used", "drs_date_note"],
)
drs["drs_date_verified"] = False   # flip to True per country once checked

# Reloop collection rates for deposit-covered containers (slide 85 of the deck).
# Undated, advocacy source: use as descriptive context only, not as an outcome.
reloop_collection = {"DK": 90, "EE": 87, "FI": 92, "DE": 98, "IS": 87,
                     "LT": 92, "NL": 95, "NO": 88, "SE": 85}
drs["reloop_collection_rate_pct"] = drs["geo"].map(reloop_collection)
drs

,geo,drs_start_deck,drs_start_used,drs_date_note,drs_date_verified,reloop_collection_rate_pct
0,SE,1984.0,1984,Deck: 1984 (cans first; check when PET joined),False,85.0
1,IS,1989.0,1989,,False,87.0
2,FI,1996.0,1996,,False,92.0
3,NO,1999.0,1999,,False,88.0
4,DK,2002.0,2002,,False,90.0
5,DE,2003.0,2003,,False,98.0
6,EE,2005.0,2005,,False,87.0
7,NL,2005.0,2005,Initially large PET bottles only; check when s...,False,95.0
8,HR,2006.0,2006,,False,NaN
9,LT,2016.0,2016,In-window switcher,False,92.0


## 3. Assemble the country–year panel

In [13]:
country_names = {k: v for k, v in rates_labels["geo"].items()}

geos = [g for g in rates_labels["geo"] if g not in DROP_GEOS]
years = sorted(rates["year"].unique())

# Full grid so that missing country-years show up explicitly as NaN rather than disappearing
panel = pd.MultiIndex.from_product([geos, years], names=["geo", "year"]).to_frame(index=False)
panel["country"] = panel["geo"].map(country_names)

panel = (
    panel.merge(rates, on=["geo", "year"], how="left")
         .merge(waste, on=["geo", "year"], how="left")
         .merge(drs, on="geo", how="left")
)

# --- DRS variables --------------------------------------------------------------------
# Countries not in the table never had a DRS during 2014-2023 (as far as our sources say)
panel["has_drs_listed"] = panel["drs_start_used"].notna()
panel["has_drs"] = (panel["year"] >= panel["drs_start_used"]).fillna(False).astype(int)
panel["drs_full_year"] = (panel["year"] > panel["drs_start_used"]).fillna(False).astype(int)
panel["years_since_drs"] = np.where(
    panel["drs_start_used"].notna() & (panel["drs_start_used"] <= years[-1]),
    panel["year"] - panel["drs_start_used"],
    np.nan,
)   # negative = before introduction, NaN = never treated in the window

def drs_group(start):
    if pd.isna(start) or start > years[-1]:
        return "3_no_drs_in_window"
    if start < years[0]:
        return "1_drs_before_window"
    return "2_drs_introduced_in_window"

panel["drs_group"] = panel["drs_start_used"].apply(drs_group)

# --- Derived outcome / context variables ----------------------------------------------
panel["rate_recomputed"] = panel["plastic_recycled_kg_hab"] / panel["plastic_generated_kg_hab"] * 100
abroad = panel["recycled_other_eu_kg_hab"] + panel["recycled_outside_eu_kg_hab"]
panel["share_recycled_abroad_pct"] = abroad / panel["plastic_recycled_kg_hab"] * 100

panel["series_break_flag"] = panel["rate_flag"].isin(["b", "bd", "be"])
panel["post_2020_method_period"] = (panel["year"] >= 2020).astype(int)   # reporting-method caution, see notes

panel = panel.sort_values(["geo", "year"]).reset_index(drop=True)
panel.head(12)

,geo,year,country,plastic_recycling_rate,rate_flag,plastic_generated_kg_hab,plastic_recycled_kg_hab,recycled_in_country_kg_hab,recycled_other_eu_kg_hab,recycled_outside_eu_kg_hab,...,reloop_collection_rate_pct,has_drs_listed,has_drs,drs_full_year,years_since_drs,drs_group,rate_recomputed,share_recycled_abroad_pct,series_break_flag,post_2020_method_period
0,AT,2014,Austria,33.6,,34.16,11.47,NaN,NaN,NaN,...,NaN,True,0,0,NaN,3_no_drs_in_window,33.577283,NaN,False,0
1,AT,2015,Austria,33.6,,34.12,11.46,NaN,NaN,NaN,...,NaN,True,0,0,NaN,3_no_drs_in_window,33.587339,NaN,False,0
2,AT,2016,Austria,33.6,,34.09,11.45,NaN,NaN,NaN,...,NaN,True,0,0,NaN,3_no_drs_in_window,33.587562,NaN,False,0
3,AT,2017,Austria,33.4,,34.36,11.48,NaN,NaN,NaN,...,NaN,True,0,0,NaN,3_no_drs_in_window,33.410943,NaN,False,0
4,AT,2018,Austria,31.9,,34.16,10.89,10.35,0.54,0.00,...,NaN,True,0,0,NaN,3_no_drs_in_window,31.879391,4.958678,False,0
5,AT,2019,Austria,30.8,,33.31,10.25,9.99,0.26,0.00,...,NaN,True,0,0,NaN,3_no_drs_in_window,30.771540,2.536585,False,0
6,AT,2020,Austria,25.3,b,33.55,8.49,7.46,1.03,0.00,...,NaN,True,0,0,NaN,3_no_drs_in_window,25.305514,12.131920,True,1
7,AT,2021,Austria,25.9,,33.67,8.72,7.67,1.05,0.00,...,NaN,True,0,0,NaN,3_no_drs_in_window,25.898426,12.041284,False,1
8,AT,2022,Austria,25.4,,33.17,8.43,6.77,1.50,0.16,...,NaN,True,0,0,NaN,3_no_drs_in_window,25.414531,19.691578,False,1
9,AT,2023,Austria,26.9,,32.29,8.69,6.23,2.19,0.27,...,NaN,True,0,0,NaN,3_no_drs_in_window,26.912357,28.308400,False,1


## 4. Sanity checks

In [14]:
# 1. One row per country-year
assert not panel.duplicated(["geo", "year"]).any()
print("rows:", len(panel), "| countries:", panel["geo"].nunique(), "| years:", years[0], "-", years[-1])

# 2. The official rate should match recycled / generated (it is that ratio)
diff = (panel["plastic_recycling_rate"] - panel["rate_recomputed"]).abs()
print(f"official vs recomputed rate: median diff {diff.median():.2f} pp, max {diff.max():.2f} pp")
print("country-years differing by more than 1 pp:")
display(panel.loc[diff > 1, ["geo", "year", "plastic_recycling_rate", "rate_recomputed", "rate_flag"]])

# 3. Missingness of the outcome
miss = panel[panel["plastic_recycling_rate"].isna()][["geo", "year"]]
print("country-years with no recycling rate:", len(miss))
display(miss.groupby("geo")["year"].apply(list))

# 4. Group sizes
display(panel.groupby("drs_group")["geo"].nunique().rename("n_countries"))
display(panel[panel["drs_group"] == "2_drs_introduced_in_window"]
        .groupby("geo")[["drs_start_used"]].first())

rows: 300 | countries: 30 | years: 2014 - 2023
official vs recomputed rate: median diff 0.02 pp, max 0.07 pp
country-years differing by more than 1 pp:


,geo,year,plastic_recycling_rate,rate_recomputed,rate_flag


country-years with no recycling rate: 8


geo
BG          [2020, 2023]
EL                [2020]
HR                [2023]
HU                [2023]
PL    [2020, 2021, 2023]
Name: year, dtype: object

drs_group
1_drs_before_window            9
2_drs_introduced_in_window     5
3_no_drs_in_window            16
Name: n_countries, dtype: int64

,drs_start_used
geo,
LT,2016.0
LV,2022.0
MT,2022.0
RO,2023.0
SK,2022.0


## 5. Save

Two files: the panel itself, and a data dictionary so future-you (and a marker) can see what each column is.

In [16]:
column_order = [
    "geo", "country", "year",
    # outcome
    "plastic_recycling_rate", "rate_flag", "series_break_flag", "post_2020_method_period",
    # volumes
    "plastic_generated_kg_hab", "plastic_recycled_kg_hab", "rate_recomputed",
    "recycled_in_country_kg_hab", "recycled_other_eu_kg_hab", "recycled_outside_eu_kg_hab",
    "share_recycled_abroad_pct",
    # treatment
    "has_drs", "drs_full_year", "years_since_drs", "drs_group",
    "drs_start_used", "drs_start_deck", "drs_date_verified", "drs_date_note",
    "reloop_collection_rate_pct",
]
final = panel[column_order]
final.to_csv(OUT_DIR / "drs_recycling_panel.csv", index=False, encoding="utf-8")

dictionary = pd.DataFrame(
    [
        ("geo", "Eurostat geo code (2 letters)", "Eurostat"),
        ("country", "Country name", "Eurostat"),
        ("year", "Calendar year, 2014-2023", "Eurostat"),
        ("plastic_recycling_rate", "Plastic packaging recycling rate, % (2025-target basis)", "Eurostat ENV_WASPACR"),
        ("rate_flag", "Eurostat status flag for the rate (b/bd/be = break in series, d = definition differs, e = estimated, p = provisional, i = imputed)", "Eurostat"),
        ("series_break_flag", "True if rate_flag signals a break in the time series", "derived"),
        ("post_2020_method_period", "1 for 2020 onwards. Many countries show a drop in 2020, probably linked to a change in the reporting method (check Eurostat metadata)", "derived"),
        ("plastic_generated_kg_hab", "Plastic packaging waste generated, kg per inhabitant", "Eurostat ENV_WASPAC"),
        ("plastic_recycled_kg_hab", "Plastic packaging waste recycled, kg per inhabitant", "Eurostat ENV_WASPAC"),
        ("rate_recomputed", "recycled / generated x 100, to cross-check the official rate", "derived"),
        ("recycled_in_country_kg_hab", "Recycled inside the reporting country (partial coverage)", "Eurostat ENV_WASPAC"),
        ("recycled_other_eu_kg_hab", "Recycled in another EU member state (partial coverage)", "Eurostat ENV_WASPAC"),
        ("recycled_outside_eu_kg_hab", "Recycled outside the EU (partial coverage)", "Eurostat ENV_WASPAC"),
        ("share_recycled_abroad_pct", "Share of recycled plastic packaging recycled outside the country, %", "derived"),
        ("has_drs", "1 if year >= DRS start year used, else 0", "hand-coded"),
        ("drs_full_year", "1 if year > DRS start year (first full year of operation)", "hand-coded"),
        ("years_since_drs", "year - DRS start year; negative = before; empty = no DRS in window", "derived"),
        ("drs_group", "1 = DRS before 2014, 2 = DRS introduced 2014-2023, 3 = no DRS in window", "derived"),
        ("drs_start_used", "DRS start year used in this analysis", "hand-coded, TO VERIFY"),
        ("drs_start_deck", "DRS start year according to the Reloop deck", "Reloop deck"),
        ("drs_date_verified", "Whether the start year has been checked against a primary source", "hand-coded"),
        ("drs_date_note", "Free-text note on the DRS date", "hand-coded"),
        ("reloop_collection_rate_pct", "Collection rate of deposit-covered containers (undated, advocacy source; context only)", "Reloop deck, slide 85"),
    ],
    columns=["column", "description", "source"],
)
dictionary.to_csv(OUT_DIR / "drs_recycling_panel_dictionary.csv", index=False, encoding="utf-8")
print("Saved", OUT_DIR / "drs_recycling_panel.csv", final.shape)

Saved data/drs_recycling_panel.csv (300, 23)


## 6. Quick first look (starting point for your univariate / bivariate work)

In [17]:
final["plastic_recycling_rate"].describe()

count    292.000000
mean      39.426712
std       11.989106
min        6.800000
25%       30.775000
50%       37.950000
75%       48.025000
max       74.400000
Name: plastic_recycling_rate, dtype: float64

## Notes for the analysis

- **Unit of analysis:** country-year. Observations from the same country are not independent, which matters once you move beyond descriptives.
- **Only three clean pre/post cases** fit the window: Lithuania (2016), Latvia (2022), Slovakia (2022). Malta and Romania, if their dates hold, have at most one or two post years.
- **Outcome is all plastic packaging**, not just drink containers.
- **Series breaks:** the 2020 drop seen in many countries is probably a reporting-method change, so avoid treating a 2019 to 2020 fall as a real change. Check the Eurostat metadata.
- **Missing values are left as NaN**: nothing has been imputed.
- **Confounders not in the dataset:** GDP, landfill/EPR policies, the 2018 Chinese import ban, deposit amount. Add them as columns if you want to go beyond association.